<a href="https://colab.research.google.com/github/iislestari689/Projek/blob/main/projek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import streamlit as st
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import time
import requests
from streamlit_lottie import st_lottie

# 1. Konfigurasi Halaman & Inisialisasi State di Awal
st.set_page_config(page_title="Kalkulator BMI", page_icon="⚖️")

# Fungsi untuk memuat stiker bergerak (Lottie) dari URL
def load_lottieurl(url: str):
    r = requests.get(url)
    if r.status_code != 200:
        return None
    return r.json()

# Memuat stiker bergerak bertema kesehatan/timbangan
lottie_health = load_lottieurl("https://lottiefiles.com")
lottie_success = load_lottieurl("https://lottiefiles.com")

if "riwayat" not in st.session_state:
    st.session_state.riwayat = []

def hitung_bmi(berat, tinggi):
    return berat / (tinggi ** 2)

def kategori_bmi(bmi):
    if bmi < 18.5:
        return "Kekurangan berat badan", "#3498db"  # Biru
    elif 18.5 <= bmi < 25.0:
        return "Berat badan normal", "#2ecc71"      # Hijau
    elif 25.0 <= bmi < 30.0:
        return "Kelebihan berat badan", "#f39c12"  # Oranye
    else:
        return "Obesitas", "#e74c3c"                # Merah

# Header dengan Stiker Bergerak Utama
col_title, col_logo = st.columns([3, 1])
with col_title:
    st.title("⚖️ Kalkulator BMI Sederhana")
    st.write("Yuk cek kondisi berat badanmu dan pantau riwayatnya dalam grafik!")
with col_logo:
    if lottie_health:
        st_lottie(lottie_health, height=120, key="main_logo")

# Input pengguna
col1, col2 = st.columns(2)
with col1:
    berat = st.number_input("Berat badan (kg):", min_value=1.0, value=60.0, step=0.1)
with col2:
    tinggi_cm = st.number_input("Tinggi badan (cm):", min_value=50.0, value=165.0, step=1.0)

if st.button("🔍 Hitung BMI", use_container_width=True):
    if tinggi_cm > 0:
        # Animasi loading sebelum hasil muncul
        with st.spinner("Menghitung BMI kamu..."):
            progress_bar = st.progress(0)
            for i in range(100):
                time.sleep(0.003)
                progress_bar.progress(i + 1)
            progress_bar.empty()

        tinggi_m = tinggi_cm / 100
        bmi = hitung_bmi(berat, tinggi_m)
        kategori, warna = kategori_bmi(bmi)
        waktu_cek = datetime.now().strftime("%d-%m-%Y %H:%M")

        st.balloons()

        # Layout Hasil dengan Stiker Bergerak Sukses
        res_col1, res_col2 = st.columns([2, 1])
        with res_col1:
            st.markdown(
                f"""
                <div style="background-color:{warna}; padding:20px; border-radius:10px; text-align:center; color:white; margin-bottom:20px;">
                    <h2 style="margin:0; color:white;">BMI Anda: {bmi:.2f}</h2>
                    <h3 style="margin:5px 0; color:white; font-weight:bold;">{kategori}</h3>
                    <p style="margin:0; font-size:14px; opacity:0.9;">📅 {waktu_cek}</p>
                </div>
                """,
                unsafe_allow_html=True
            )
        with res_col2:
            if lottie_success:
                st_lottie(lottie_success, height=140, key="success_anim")

        # Simpan ke dalam riwayat session state
        st.session_state.riwayat.append({
            "waktu": waktu_cek,
            "berat": berat,
            "tinggi_cm": tinggi_cm,
            "bmi": round(bmi, 2),
            "kategori": kategori
        })
    else:
        st.error("Tinggi badan harus lebih dari 0!")

# Grafik riwayat BMI (Menampilkan diagram garis setiap pengecekan)
if st.session_state.riwayat:
    st.write("---")
    st.subheader("📈 Diagram Garis Pengecekan Berkala")

    df = pd.DataFrame(st.session_state.riwayat)
    # Membuat label sumbu X berupa urutan pengecekan dan waktu agar informatif
    df["label_pengecekan"] = "Ke-" + (df.index + 1).astype(str) + " (" + df["waktu"] + ")"

    fig = go.Figure()

    # Membuat diagram garis interaktif
    fig.add_trace(go.Scatter(
        x=df["label_pengecekan"],
        y=df["bmi"],
        mode="lines+markers+text",
        text=df["bmi"],
        textposition="top center",
        line=dict(color="#6c5ce7", width=3, shape="spline"),
        marker=dict(size=12, color="#6c5ce7", line=dict(width=2, color="white")),
        name="Nilai BMI Anda"
    ))

    # Garis batas kategori standar kesehatan (referensi visual mendatar)
    fig.add_hline(y=18.5, line_dash="dot", line_color="#3498db", annotation_text="Batas Kurus (<18.5)")
    fig.add_hline(y=25.0, line_dash="dot", line_color="#2ecc71", annotation_text="Batas Normal (18.5-25)")
    fig.add_hline(y=30.0, line_dash="dot", line_color="#e74c3c", annotation_text="Batas Obesitas (>30)")

    fig.update_layout(
        xaxis_title="Daftar Riwayat Pengecekan",
        yaxis_title="Nilai Skor BMI",
        template="plotly_white",
        height=450,
        margin=dict(l=20, r=20, t=40, b=80) # Margin bawah diperlebar agar teks tanggal tidak terpotong
    )

    st.plotly_chart(fig, use_container_width=True)

    # Tabel rincian data di bawah grafik
    with st.expander("👁️ Lihat Tabel Rincian Data"):
        st.dataframe(df[["waktu", "berat", "tinggi_cm", "bmi", "kategori"]], use_container_width=True)

    # Tombol hapus riwayat
    if st.button("🗑️ Hapus Semua Riwayat", use_container_width=True):
        st.session_state.riwayat = []
        st.rerun()

2026-06-30 07:58:54.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.697 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.710 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.713 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-30 07:58:54.728 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar